# How a climate model computes radiation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/PySoc/blob/main/notebooks/01_how_radiation_works.ipynb)

Every climate model has a **radiation scheme**: the part of the code that works out how much sunlight
is absorbed and how much heat escapes to space, layer by layer, in every column of the atmosphere.
This notebook takes one such scheme apart.

The scheme here is **SOCRATES**, the radiation code of the UK Met Office, set up exactly as the
[Isca](https://execlim.github.io/Isca/) climate model uses it. We run **PySoc**, a Python (PyTorch)
rewrite of SOCRATES that gives the same numbers as the original Fortran.

**By the end you should be able to explain:**
1. why radiation is split into *shortwave* (sunlight) and *longwave* (heat) parts;
2. what a radiation scheme needs to know about the atmosphere;
3. how fluxes of energy become heating and cooling rates;
4. why some wavelengths escape to space and others don't (the greenhouse effect);
5. how the scheme represents thousands of absorption lines with a few dozen "k-terms";
6. how clouds change the picture.

**How to use it:** run the cells from top to bottom (Shift+Enter). Cells with sliders are interactive:
move the sliders and watch the plots change. Nothing needs installing on your own computer.
The code in each cell is hidden to keep the page readable: click **Show code** (or double-click the
cell's title) to read it.

In [ ]:
#@title Setup: run this cell first (it takes about a minute on Colab)
import importlib, os, subprocess, sys

if os.path.isdir("../pysoc") and os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))  # running inside a copy of the repository
try:
    import pysoc
except ImportError:  # on Colab: install PySoc from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/evanwellmeyer/PySoc"],
                   check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import torch
from cycler import cycler
from ipywidgets import FloatSlider, IntSlider, Dropdown, interact

from pysoc.column import make_column, liquid_cloud
from pysoc.isca import IscaSocrates, GAS_NAMES, CP_AIR, GRAV, ozone_mmr_from_vmr
from pysoc.spectra import ga7_spectral_files

torch.set_flush_denormal(True)  # avoids a slowdown in float64 on CPUs
torch.set_num_threads(min(4, torch.get_num_threads()))  # one column runs fastest on a few threads
lw_file, sw_file = ga7_spectral_files()  # downloads the SOCRATES GA7 spectral files once
model = IscaSocrates(lw_file, sw_file)

# plot style: one fixed colour per role, thin lines, quiet grid
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LW_COLOR, SW_COLOR, NET_COLOR = BLUE, ORANGE, INK
plt.rcParams.update({
    "figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "lines.linewidth": 2, "legend.frameon": False,
    "axes.prop_cycle": cycler(color=[BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]),
})


def radiation(col, albedo=0.3, insolation=340.0, coszen=0.5, cloud=None, remove=(), intermediates=False):
    """Run SOCRATES on a column made by make_column (or a modified copy of one).

    albedo      surface albedo (fraction of sunlight the ground reflects)
    insolation  sunlight reaching the top of the atmosphere, averaged over day and night (W/m2)
    coszen      cosine of the solar zenith angle: 1 = sun overhead, 0.5 = sun 60 degrees from overhead
    cloud       cloud fields from liquid_cloud(...)
    remove      names of gases to leave out, e.g. ("CO2", "H2O")
    """
    ids = {name: i for i, name in GAS_NAMES.items()}
    model.config.exclude_gases = frozenset(ids[name] for name in remove)
    try:
        # SOCRATES takes the solar constant and the sun's angle; scale the sun so the average is `insolation`
        rrsun = insolation / (coszen * model.config.stellar_constant)
        return model(**col, albedo=albedo, coszen=coszen, rrsun=rrsun, delta_t=0.0,
                     return_intermediates=intermediates, **(cloud or {}))
    finally:
        model.config.exclude_gases = frozenset()


def pressure_axis(ax, top=0.01):
    ax.set_yscale("log")
    ax.set_ylim(1000, top)
    ax.set_yticks([t for t in (1000, 300, 100, 30, 10, 3, 1, 0.3, 0.1, 0.03, 0.01) if t >= top])
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.get_yaxis().set_minor_formatter(plt.NullFormatter())
    ax.set_ylabel("pressure (hPa)")


hpa = lambda p: (p / 100).numpy()  # noqa: E731  Pa -> hPa, as numpy
per_day = lambda rate: (rate * 86400).detach().numpy()  # noqa: E731  K/s -> K/day
print("Ready: SOCRATES loaded with the", os.path.basename(lw_file), "and", os.path.basename(sw_file), "spectral files.")

## 1. Two kinds of radiation

Everything emits radiation, with a spectrum set by its temperature (Planck's law). The Sun (about 5800 K)
emits mostly visible and near-infrared light; the Earth and its atmosphere (about 200–300 K) emit in
the thermal infrared. The two spectra barely overlap, so climate models treat them as two separate
problems:

* **shortwave (SW)**: sunlight, which is scattered and absorbed on its way down and back up;
* **longwave (LW)**: thermal infrared, emitted by the surface and by every layer of the atmosphere.

SOCRATES splits each part into **bands**: 6 shortwave and 9 longwave. The strips under the curves
show them. (Two longwave bands sit *inside* others: band 4 carves the 15 µm CO₂ band out of band 3,
and band 6 carves the 9.6 µm ozone band out of band 5.)

The curves show energy per *logarithmic* wavelength interval, so equal areas mean equal energy on this
axis. (Their peaks sit a little longer than the textbook Wien's-law peak, which is per unit wavelength.)

**Try it:** change the temperatures. Where does a 3000 K star (a red dwarf) put its light? What about
a planet at 400 K?

In [ ]:
#@title Sunlight and Earth's emission (interactive: change the temperatures)
H, C, K_B = 6.626e-34, 2.998e8, 1.381e-23
wavelength = np.logspace(-1, 2, 1000)  # 0.1 to 100 micrometres


def planck(wavelength_um, temperature):
    """Blackbody emission per unit wavelength, pi*B (W/m2 per micrometre)."""
    lam = wavelength_um * 1e-6
    return np.pi * 2 * H * C**2 / lam**5 / np.expm1(H * C / (lam * K_B * temperature)) * 1e-6


sw_spec, lw_spec = model.sw.spectrum.sp, model.lw.spectrum.sp


def draw_bands(ax, sp, y, height, color, label):
    excluded = sp.index_exclude or [[] for _ in range(sp.n_band)]
    nested = {j for excl in excluded for j in excl}
    for b in range(sp.n_band):
        lo, hi = sp.wavelength_short[b] * 1e6, min(sp.wavelength_long[b] * 1e6, 100)
        inner = b in nested  # a band nested inside another
        ax.add_patch(plt.Rectangle((lo, y + (0.2 if inner else 0) * height), hi - lo, height * (0.6 if inner else 1),
                                   facecolor=color, alpha=0.35 if inner else 0.18, edgecolor="white", linewidth=1.5,
                                   transform=ax.get_xaxis_transform(), clip_on=False))
        # put the label in the widest part of the band that no nested band covers
        segments, start = [], lo
        for j in sorted(excluded[b], key=lambda j: sp.wavelength_short[j]):
            segments.append((start, sp.wavelength_short[j] * 1e6))
            start = sp.wavelength_long[j] * 1e6
        segments.append((start, hi))
        a, z = max(segments, key=lambda s: np.log(s[1] / s[0]))
        ax.text(np.sqrt(a * z), y + height / 2, str(b + 1), transform=ax.get_xaxis_transform(),
                ha="center", va="center", fontsize=8, color=INK2)
    ax.text(0.1, y + height / 2, label + "  ", transform=ax.get_xaxis_transform(), ha="right", va="center",
            fontsize=9, color=INK2)


def plot_spectra(t_sun=5772, t_earth=288):
    sun = planck(wavelength, t_sun) * (6.957e8 / 1.496e11) ** 2  # sunlight arriving at the Earth
    earth = planck(wavelength, t_earth)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    for spectrum, color, name in ((sun, SW_COLOR, f"Sun ({t_sun} K)"), (earth, LW_COLOR, f"Earth ({t_earth} K)")):
        shape = wavelength * spectrum  # per log-wavelength, so areas compare fairly
        ax.fill_between(wavelength, shape / shape.max(), color=color, alpha=0.25, linewidth=0)
        ax.plot(wavelength, shape / shape.max(), color=color, label=name)
        peak = wavelength[np.argmax(shape)]
        ax.annotate(f"peak {peak:.2g} µm", (peak, 1.0), xytext=(0, 6), textcoords="offset points",
                    ha="center", fontsize=9, color=INK2)
    ax.set_xscale("log")
    ax.set_xlim(0.1, 100)
    ax.set_ylim(0, 1.15)
    ax.set_xlabel("wavelength (µm)")
    ax.set_ylabel("emission, scaled to its peak")
    ax.set_title("Sunlight and Earth's heat barely overlap")
    ax.legend(loc="upper right")
    draw_bands(ax, sw_spec, -0.30, 0.09, SW_COLOR, "SW bands")
    draw_bands(ax, lw_spec, -0.42, 0.09, LW_COLOR, "LW bands")
    fig.subplots_adjust(bottom=0.36, left=0.14)
    plt.show()
    total = lambda y, x: np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2)  # noqa: E731  trapezoid rule
    cut = wavelength > 4.0
    sun_above = total(sun[cut], wavelength[cut]) / total(sun, wavelength)
    earth_below = total(earth[~cut], wavelength[~cut]) / total(earth, wavelength)
    print(f"Sunlight at wavelengths longer than 4 µm: {100 * sun_above:.2f}% of the total")
    print(f"Earth's emission at wavelengths shorter than 4 µm: {100 * earth_below:.4f}% of the total")


interact(plot_spectra, t_sun=IntSlider(5772, 2500, 10000, 100, continuous_update=True),
         t_earth=IntSlider(288, 150, 450, 5, continuous_update=True));

## 2. The model atmosphere: one column

A radiation scheme works on one **column** of atmosphere at a time. A climate model has thousands of
columns (Isca at its standard resolution has 8192) and calls the scheme for each of them.

`make_column` builds an idealised column like the ones Isca passes to SOCRATES. It has 40 layers on
Isca's stretched grid: thin layers high up and thicker ones near the ground. Index 0 is the **top**
of the atmosphere. Each layer has a temperature and gas amounts; the boundaries between layers are
called **half levels** (41 of them), and that is where the fluxes are computed.

The profiles are textbook choices: temperature falls by 6.5 K per km up to the tropopause, then
follows the US Standard Atmosphere; relative humidity decreases with height; ozone peaks in the
stratosphere with a total of 300 Dobson units; CO₂ is 280 ppm (pre-industrial).

In [ ]:
#@title Build the model column
col = make_column(t_surf=288.0, co2_ppmv=280.0)
for name, value in col.items():
    print(f"{name:7s} shape {str(tuple(value.shape)):8s}  first (top) value {value.reshape(-1)[0].item():.4g}")

**Try it:** the sliders change the column. The grey lines are the column above (the **control**), and
each dot is one of the 40 model layers. Below the plots, the model's energy budget for your column is
compared with the control's (section 3 explains these numbers).

In [ ]:
#@title Explore the column (interactive: surface temperature, lapse rate, humidity, ozone)
p = hpa(col["p_full"])
o_col = radiation(col)


def explore_column(t_surf=288.0, lapse_rate=6.5, rh_surface=0.8, ozone_du=300.0):
    new = make_column(t_surf=t_surf, lapse_rate=lapse_rate, rh_surface=rh_surface, ozone_du=ozone_du, co2_ppmv=280.0)
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.2), sharey=True)
    fields = (("temp", 1.0, RED, "temperature (K)", "Temperature"),
              ("q", 1000.0, BLUE, "specific humidity (g/kg)", "Water vapour"),
              ("ozone", 1e6 / ozone_mmr_from_vmr(1.0), AQUA, "ozone (ppmv)", "Ozone"))
    for ax, (key, scale, color, xlabel, title) in zip(axes, fields):
        ax.plot(scale * col[key].numpy(), p, color=MUTED, linewidth=1.5, label="control")
        ax.plot(scale * new[key].numpy(), p, marker="o", markersize=3, color=color, label="your column")
        ax.set_xlabel(xlabel)
        ax.set_title(title)
    axes[1].set_xscale("log")
    axes[0].legend(loc="upper right")
    pressure_axis(axes[0])
    for ax in axes[1:]:
        ax.set_ylabel("")
    fig.tight_layout()
    plt.show()
    o = radiation(new)
    t = new["temp"].numpy()
    k = len(t) - 1
    while k > 0 and t[k - 1] < t[k] - 1e-6:  # going up from the surface while the temperature still falls
        k -= 1
    print(f"tropopause (where the temperature stops falling with height): {t[k]:.1f} K at {p[k]:.0f} hPa")
    for label, key in (("outgoing longwave (OLR)", "soc_olr"), ("sunlight absorbed", "soc_toa_sw")):
        print(f"{label:24s} control {float(o_col[key]):6.1f} W/m²   your column {float(o[key]):6.1f} W/m²"
              f"   change {float(o[key] - o_col[key]):+6.1f} W/m²")


interact(explore_column,
         t_surf=FloatSlider(288, min=260, max=320, step=1, continuous_update=True),
         lapse_rate=FloatSlider(6.5, min=3, max=9.5, step=0.5, continuous_update=True),
         rh_surface=FloatSlider(0.8, min=0.0, max=1.0, step=0.05, continuous_update=True),
         ozone_du=FloatSlider(300, min=0, max=600, step=25, continuous_update=True));

**Try it and describe**
1. Warm the surface to 298 K, then cool it to 278 K. Describe how the temperature profile, the tropopause
   (printed below the plots) and the water vapour respond. How does the OLR change?
2. Change the lapse rate. Describe how the upper troposphere and the OLR respond.
3. Change the surface humidity, then the ozone. Which parts of the column change? What happens to the OLR
   and to the absorbed sunlight?

**Explain:** water vapour falls by a factor of about 1000 between the surface and the tropopause. Why?
(Hint: how much water vapour can cold air hold?)

## 3. Running the radiation code

One call computes everything. `radiation` (defined in the setup cell) passes the column to SOCRATES,
together with the surface albedo and the sunlight. For sunlight we use the **global average**: 340 W/m²
arriving at the top of the atmosphere (a quarter of the solar constant, because the Earth is a sphere
and half of it is in darkness at any time).

There are no clouds yet, and the surface albedo of 0.3 is higher than the real ocean's (about 0.06).
Simple models often do this so the missing clouds' reflection is roughly accounted for.

In [ ]:
#@title Run the radiation code and print the energy budget
out = radiation(col)

f = lambda key: float(out[key])  # noqa: E731
print("Top of the atmosphere")
print(f"  sunlight in                     {f('soc_toa_sw_down'):6.1f} W/m²")
print(f"  sunlight reflected to space     {f('soc_toa_sw_up'):6.1f} W/m²")
print(f"  outgoing longwave (OLR)         {f('soc_olr'):6.1f} W/m²")
print(f"  net (in - out)                  {f('soc_toa_sw') - f('soc_olr'):6.1f} W/m²")
print("Atmosphere")
print(f"  sunlight absorbed               {f('soc_toa_sw') - f('soc_surf_flux_sw'):6.1f} W/m²")
print("Surface")
print(f"  sunlight absorbed               {f('soc_surf_flux_sw'):6.1f} W/m²")
print(f"  longwave emitted (sigma*Ts^4)   {float(out['flux_lw_up'][-1]):6.1f} W/m²")
print(f"  longwave from the sky           {f('soc_surf_flux_lw_down'):6.1f} W/m²")

The outputs are named as in Isca (`soc_olr`, `soc_toa_sw`, …). The full profiles are there too.
Fluxes are defined on the half levels; `_up` and `_down` are the upward and downward streams.
SOCRATES is a **two-stream** scheme: it tracks just these two streams, rather than radiation in every
direction.

In [ ]:
#@title Plot the longwave and shortwave fluxes
ph = hpa(col["p_half"])
ph[0] = 0.01  # the top half level is at p = 0; draw it at the top of the axis
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), sharey=True)
axes[0].plot(out["flux_lw_up"].numpy(), ph, color=LW_COLOR, label="upward")
axes[0].plot(out["flux_lw_down"].numpy(), ph, color=LW_COLOR, linestyle="--", label="downward")
axes[0].set_title("Longwave")
axes[1].plot(out["flux_sw_up"].numpy(), ph, color=SW_COLOR, label="upward")
axes[1].plot(out["flux_sw_down"].numpy(), ph, color=SW_COLOR, linestyle="--", label="downward")
axes[1].set_title("Shortwave")
for ax in axes:
    ax.set_xlabel("flux (W/m²)")
    ax.legend(loc="center right")
pressure_axis(axes[0])
fig.tight_layout()
plt.show()

**Describe what you see**
1. Describe how the upward longwave flux changes from the surface to the top of the atmosphere. Where does
   it change fastest?
2. Describe the downward longwave flux: where is it largest, and where is it zero? Where does the downward
   longwave at the surface come from?
3. Describe the shortwave fluxes on the way down and back up. Is this column in energy balance at the top
   of the atmosphere? (In Notebook 2 a column finds its own balance.)

## 4. From fluxes to heating rates

Radiation heats or cools a layer when more energy flows in than out. With the **net flux**
$F = F_{up} - F_{down}$ (positive upward), a layer between pressures $p_{top}$ and $p_{bottom}$ holds
a mass $\Delta p / g$ of air per square metre, and its temperature changes at the rate

$$\frac{dT}{dt} = \frac{g}{c_p}\,\frac{F(p_{bottom}) - F(p_{top})}{\Delta p}$$

where $c_p$ is the heat capacity of air. Let's compute it by hand and compare with the model's own
heating rate, `tdt_lw` (in K per second).

In [ ]:
#@title Heating rates by hand, compared with the model
F_lw = out["flux_lw_up"] - out["flux_lw_down"]
dp = col["p_half"][1:] - col["p_half"][:-1]
heating_by_hand = GRAV * (F_lw[1:] - F_lw[:-1]) / (CP_AIR * dp)
print("largest difference from the model's tdt_lw:", float((heating_by_hand - out["tdt_lw"]).abs().max()), "K/s")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.axvline(0, color=AXIS, linewidth=1)
ax.plot(per_day(out["tdt_lw"]), p, color=LW_COLOR, label="longwave")
ax.plot(per_day(out["tdt_sw"]), p, color=SW_COLOR, label="shortwave")
ax.plot(per_day(out["tdt_rad"]), p, color=NET_COLOR, label="total")
pressure_axis(ax)
ax.set_xlabel("heating rate (K/day)")
ax.set_title("Radiative heating and cooling")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

**Describe what you see**
1. Describe where radiation heats the column and where it cools it, and by roughly how much.
2. Describe the shortwave heating in the stratosphere. Where does it peak? Compare with the ozone profile in
   section 2.

**Explain:** the troposphere (below about 200 hPa) cools by 1–2 K per day, yet it doesn't get colder every
day. What heats it? (Think about what happens after it rains, and about warm air rising from the ground.)

## 5. Where the longwave escapes: the greenhouse effect, band by band

The surface at 288 K emits about 390 W/m². Only about 264 W/m² leaves the top of the atmosphere.
The difference is the **greenhouse effect**. It is not the same at all wavelengths. For each longwave
band the bars show what the surface emits (grey outline) and what actually escapes to space (blue).

**Try it:** switch gases off. The model then runs without them (it keeps the temperatures the same).
Which gas matters most? In which bands? What happens to the "window" band (5)?

In [ ]:
#@title Outgoing longwave radiation band by band (interactive: switch gases off)
BAND_NICKNAMES = ["far IR (H₂O)", "H₂O", "CO₂ wings, H₂O", "CO₂ 15 µm", "window", "O₃ 9.6 µm",
                  "H₂O, CH₄, N₂O", "H₂O 6.3 µm", "H₂O, near IR"]


def band_label(sp, b):
    lo, hi = sp.wavelength_short[b] * 1e6, sp.wavelength_long[b] * 1e6
    return f"{b + 1}: {lo:.3g}–{hi:.3g} µm" if hi < 1000 else f"{b + 1}: {lo:.3g}–{hi:.0f} µm"


def plot_olr_spectrum(H2O=True, CO2=True, O3=True, other_gases=True):
    removed = [name for name, keep in (("H2O", H2O), ("CO2", CO2), ("O3", O3)) if not keep]
    if not other_gases:
        removed += ["N2O", "CH4", "CFC-11", "CFC-12", "CFC-113", "HCFC-22", "HFC-134a"]
    o = radiation(col, remove=removed, intermediates=True)
    surface = o["lw_intermediates"]["planck_ground"][:, 0].numpy()  # blackbody at the surface, per band
    olr = o["soc_spectral_olr"].numpy()
    y = np.arange(len(olr))
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    ax.barh(y, surface, height=0.7, facecolor="none", edgecolor=MUTED, linewidth=1.5, label="emitted by the surface")
    ax.barh(y, olr, height=0.7, color=LW_COLOR, label="escaping to space")
    ax.set_yticks(y, [f"{band_label(lw_spec, b)}  {BAND_NICKNAMES[b]}" for b in y])
    ax.invert_yaxis()
    ax.set_xlabel("flux in the band (W/m²)")
    ax.set_title("Outgoing longwave radiation, band by band")
    ax.legend(loc="lower right")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    plt.show()
    print(f"surface emission {surface.sum():.1f} W/m²   OLR {olr.sum():.1f} W/m²   "
          f"greenhouse effect {surface.sum() - olr.sum():.1f} W/m²"
          + (f"   (without {', '.join(removed)})" if removed else ""))


interact(plot_olr_spectrum);

**Try it and describe**
1. With all gases on, describe which bands let the most radiation escape and which the least.
2. Switch off water vapour. Describe which bands respond and by how much, and how the total OLR changes.
   Do the same for CO₂, then for ozone.
3. Switch off water vapour and CO₂ together. Is the change in OLR the sum of the two separate changes?

**Explain:** why is band 5 called the "window"? Why is the combined effect of water vapour and CO₂ not the
sum of their separate effects? (Hint: the gases *overlap* in some bands.)

## 6. Inside a band: k-terms

A real absorption spectrum has thousands of sharp lines. Computing radiation at every wavelength
would be far too slow for a climate model, which calls the scheme millions of times. SOCRATES uses
the **correlated-k method**. Within a band it sorts the wavelengths by how strongly they absorb and
groups them into a few **k-terms**. Each k-term has one absorption strength and a **weight**: the
fraction of the band it stands for. SOCRATES then solves the two-stream equations once per k-term:
81 times in the longwave and 41 in the shortwave. (PySoc solves all of them at once, which is what
makes it fast on a GPU.)

A useful way to picture a k-term is its **emission level**: roughly, the height from which its
radiation escapes to space. That is where the atmosphere above becomes transparent, at an optical
depth of about 1. (Longwave radiation travels at a slant, so SOCRATES multiplies vertical optical
depths by 1.66.) Weakly absorbing k-terms see all the way down to the surface. Strongly absorbing
ones only see the cold upper atmosphere, so they emit less.

**Try it:** look at band 4 (the CO₂ band), then band 5 (the window) and band 9 (water vapour).

In [ ]:
#@title Where each k-term emits to space (interactive: pick a band)
o6 = radiation(col, intermediates=True)
tau = o6["lw_intermediates"]["tau"][:, 0, :].numpy()  # (k-term, layer) vertical optical depth of each layer
gp_band = model.lw.spectrum.gp_band.numpy()
gp_weight = model.lw.spectrum.gp_weight.numpy()
p_half = col["p_half"].numpy()
log_p_full, temp_full = np.log(col["p_full"].numpy()), col["temp"].numpy()
print("k-terms per longwave band:", np.bincount(gp_band))


def emission_levels(band):
    """Pressure and temperature where the slant optical depth from the top reaches 1, for each k-term."""
    rows = []
    for g in np.where(gp_band == band)[0]:
        depth = np.concatenate([[0.0], np.cumsum(1.66 * tau[g])])  # at each half level, from the top
        if depth[-1] < 1.0:
            rows.append((gp_weight[g], depth[-1], p_half[-1], float(col["t_surf"])))  # sees the surface
            continue
        i = np.searchsorted(depth, 1.0)
        x = (1.0 - depth[i - 1]) / (depth[i] - depth[i - 1])
        pe = p_half[i - 1] + x * (p_half[i] - p_half[i - 1])
        rows.append((gp_weight[g], depth[-1], pe, np.interp(np.log(pe), log_p_full, temp_full)))
    return np.array(rows)


def plot_k_terms(band=4):
    b = band - 1
    rows = emission_levels(b)
    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    ax.plot(col["temp"].numpy(), p, color=AXIS, linewidth=2, label="temperature profile")
    ax.scatter(rows[:, 3], np.maximum(rows[:, 2] / 100, 0.012), s=30 + 900 * rows[:, 0], color=LW_COLOR, alpha=0.75,
               edgecolor="white", linewidth=1.5, zorder=3, label="k-term (size = weight)")
    pressure_axis(ax)
    ax.set_xlabel("temperature (K)")
    ax.set_title(f"Where band {band} ({BAND_NICKNAMES[b]}) emits to space")
    ax.legend(loc="upper right", markerscale=0.5)
    fig.tight_layout()
    plt.show()
    print(" weight   column optical depth   emission level (hPa)   temperature there (K)")
    for w, depth, pe, te in rows:
        print(f"{w:7.3f}   {depth / 1.66:20.3g}   {pe / 100:20.1f}   {te:21.1f}")


interact(plot_k_terms, band=Dropdown(options=list(range(1, 10)), value=4));

**Try it and describe**
1. For band 4 (CO₂), describe where its k-terms emit from, and which of them carry most of the band's weight.
2. Switch to band 5 (the window) and band 9 (water vapour). Describe how their emission levels differ from
   band 4's.
3. For each band, compare the temperatures at the emission levels with the surface temperature (288 K).
   Connect this to the greenhouse effect you saw in section 5.

**Explain:** the strongest CO₂ k-terms emit from the stratosphere, where it is *warmer* than at the
tropopause. With more CO₂ their emission levels would move higher still. Would they then send more or less
energy to space? (Notebook 2 lets you test this.)

## 7. Sunlight: scattering, absorption and the angle of the sun

Shortwave radiation is **scattered** (by air molecules, which is why the sky is blue, and by clouds)
and **absorbed** (mainly by ozone, water vapour and the surface). For each shortwave band the bars
show where the incoming sunlight ends up.

**Try it:** lower the sun (a smaller cos(zenith) means a lower sun and a longer path through the air)
and change the surface albedo (0.06 for ocean, 0.3 for grassland or desert, 0.8 for fresh snow).
The total daily sunlight stays at 340 W/m², so only the path and the surface change.

In [ ]:
#@title Where the sunlight goes (interactive: sun angle and surface albedo)
def plot_shortwave(coszen=0.5, albedo=0.3):
    o = radiation(col, coszen=coszen, albedo=albedo)
    down, up = o["flux_sw_down_band"].numpy(), o["flux_sw_up_band"].numpy()
    incoming = down[0]
    reflected = up[0] / incoming
    surface = (down[-1] - up[-1]) / incoming
    atmosphere = 1 - reflected - surface
    y = np.arange(sw_spec.n_band)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.2), gridspec_kw=dict(width_ratios=[1.6, 1]))
    left = np.zeros_like(reflected)
    for share, color, name in ((reflected, BLUE, "reflected to space"), (atmosphere, ORANGE, "absorbed by the air"),
                               (surface, AQUA, "absorbed by the surface")):
        ax.barh(y, share, left=left, height=0.7, color=color, edgecolor="white", linewidth=2, label=name)
        left = left + share
    ax.set_yticks(y, [band_label(sw_spec, b) + f"\n{incoming[b]:.0f} W/m² in" for b in y])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("fraction of the sunlight in each band")
    ax.set_title("Where the sunlight goes")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3)
    ax2.axvline(0, color=AXIS, linewidth=1)
    ax2.plot(per_day(o["tdt_sw"]), p, color=SW_COLOR)
    pressure_axis(ax2)
    ax2.set_xlabel("shortwave heating (K/day)")
    ax2.set_title("Heating by sunlight")
    fig.tight_layout()
    plt.show()
    print(f"planetary albedo {float(o['soc_toa_sw_up'] / o['soc_toa_sw_down']):.3f}   absorbed: "
          f"atmosphere {float(o['soc_toa_sw'] - o['soc_surf_flux_sw']):.1f} W/m², "
          f"surface {float(o['soc_surf_flux_sw']):.1f} W/m²")


interact(plot_shortwave, coszen=FloatSlider(0.5, min=0.05, max=1.0, step=0.05, continuous_update=True),
         albedo=FloatSlider(0.3, min=0.0, max=0.9, step=0.05, continuous_update=True));

**Try it and describe**
1. Set the albedo to 0.06 (ocean). Describe which bands are reflected most, which are absorbed most by the
   air, and which reach the surface.
2. Lower the sun step by step. Describe how the reflected, air-absorbed and surface-absorbed fractions
   change, and how the heating profile changes.
3. Raise the albedo to 0.8 (fresh snow). Describe the response in each band and in the planetary albedo.

**Explain:** why does the blue band (2) reflect so much more than the red one (3) over a dark ocean?
(Hint: Rayleigh scattering is much stronger at short wavelengths.) What absorbs the ultraviolet (band 1),
and where?

## 8. Clouds

Clouds reflect sunlight (cooling the planet) and absorb and re-emit longwave radiation (warming it).
Isca's simple cloud scheme gives SOCRATES liquid clouds described by three things:
* the **cloud fraction** of each layer;
* the **liquid water path**, in grams of water per square metre. About 10 g/m² is a thin, grey cloud;
  200 g/m² is a thick, bright one;
* the droplet **effective radius** (typically 5–15 µm).

The **cloud radiative effect (CRE)** measures a cloud's impact at the top of the atmosphere: the
all-sky flux minus the clear-sky flux, counted positive when the cloud warms the planet.

**Try it:** make a low, thick cloud, then a high, thin one. Which cools and which warms?

In [ ]:
#@title Clouds (interactive: height, thickness, water, droplet size, fraction)
def plot_cloud(cloud_top_hPa=750, thickness_hPa=150, liquid_water_path=100, droplet_radius=10, fraction=1.0):
    p_bottom = min(cloud_top_hPa + thickness_hPa, 1000)
    cloud = liquid_cloud(col, cloud_top_hPa * 100.0, p_bottom * 100.0, lwp=liquid_water_path,
                         fraction=fraction, reff=droplet_radius)
    o = radiation(col, cloud=cloud)
    lw_cre = float(o["soc_olr_clr"] - o["soc_olr"])
    sw_cre = float(o["soc_toa_sw"] - o["soc_toa_sw_clr"])
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw=dict(width_ratios=[1, 1.2]))
    values = [lw_cre, sw_cre, lw_cre + sw_cre]
    ax.bar([0, 1, 2], values, color=[LW_COLOR, SW_COLOR, NET_COLOR], width=0.6)
    ax.axhline(0, color=AXIS, linewidth=1)
    for x, v in enumerate(values):
        ax.annotate(f"{v:+.1f}", (x, v), xytext=(0, 4 if v >= 0 else -14), textcoords="offset points",
                    ha="center", color=INK)
    ax.set_xticks([0, 1, 2], ["longwave", "shortwave", "net"])
    ax.set_ylim(-max(160, 1.2 * max(map(abs, values))), max(160, 1.2 * max(map(abs, values))))
    ax.set_ylabel("cloud radiative effect (W/m²)")
    ax.set_title("Effect at the top of the atmosphere")
    ax.grid(axis="x", visible=False)
    ax2.axhspan(p_bottom, cloud_top_hPa, color=MUTED, alpha=0.2, linewidth=0)
    ax2.axvline(0, color=AXIS, linewidth=1)
    ax2.plot(per_day(o["tdt_lw"]), p, color=LW_COLOR, label="longwave")
    ax2.plot(per_day(o["tdt_sw"]), p, color=SW_COLOR, label="shortwave")
    ax2.plot(per_day(o["tdt_lw_clear"] + o["tdt_sw_clear"]), p, color=MUTED, linewidth=1.5, label="total, clear sky")
    ax2.set_yscale("linear")
    ax2.set_ylim(1000, 100)
    ax2.set_ylabel("pressure (hPa)")
    rates = np.concatenate([per_day(o["tdt_lw"]), per_day(o["tdt_sw"])])
    ax2.set_xlim(min(-5, 1.1 * rates.min()), max(5, 1.1 * rates.max()))
    ax2.set_xlabel("heating rate (K/day)")
    ax2.set_title("Heating inside and around the cloud (grey)")
    ax2.legend(loc="lower left")
    fig.tight_layout()
    plt.show()


interact(plot_cloud,
         cloud_top_hPa=IntSlider(750, 150, 950, 25, continuous_update=True),
         thickness_hPa=IntSlider(150, 50, 300, 25, continuous_update=True),
         liquid_water_path=FloatSlider(100, min=2, max=300, step=2, continuous_update=True),
         droplet_radius=FloatSlider(10, min=4, max=30, step=1, continuous_update=True),
         fraction=FloatSlider(1.0, min=0.0, max=1.0, step=0.05, continuous_update=True));

**Try it and describe**
1. Start with the low cloud (the default settings). Describe its longwave, shortwave and net effects, and
   the heating inside and around it.
2. Raise the cloud top to 200 hPa. Describe what changes and what stays about the same.
3. Keep the cloud high and vary the liquid water path. Describe how the net effect changes. At about what
   water path does it change sign?
4. Change the droplet radius with the water path fixed. Describe the effect. (Pollution particles make
   droplets smaller: this is one way aerosols affect climate.)

**Explain:** why does a low cloud have a small longwave effect while a high cloud has a large one? (Hint:
compare the cloud-top temperature with the surface temperature.)

## 9. Summary

* Radiation comes in two nearly separate parts: shortwave from the Sun and longwave from the Earth.
* SOCRATES divides each part into bands and each band into k-terms, then solves a two-stream problem
  for every k-term in every column.
* Differences in flux between the top and bottom of a layer heat or cool it. Radiation cools the
  troposphere, and convection and latent heating keep it from freezing.
* The greenhouse effect is strongest where absorbing gases make the air opaque, so radiation escapes
  from high, cold levels. The window near 10 µm lets surface radiation escape directly.
* Clouds cool by reflecting sunlight and warm by trapping longwave radiation; which effect wins depends
  on their height and thickness.

In Isca, this calculation runs in every column of the model at regular intervals, and the heating
rates (`tdt_rad`) feed into the model's temperature equation.

**Next:** [Notebook 2](https://colab.research.google.com/github/evanwellmeyer/PySoc/blob/main/notebooks/02_perturbation_experiments.ipynb)
uses the model for climate experiments: doubling CO₂, feedbacks, clouds and a radiative–convective
equilibrium like the one in Manabe and Wetherald's Nobel-winning work.